## Private Knowledge Worker - RAG System

### Expert Question Answerer for Raheem Yaqub Adesola

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from transformers import pipeline
from langchain_core.prompts import ChatPromptTemplate
import torch

import gradio as gr

## Define Path

In [ ]:
KNOWLEDGE_BASE = "../knowledge_base"
DB_PATH = "../vector_db"

## Load Documents

In [ ]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    #PyPDFLoader
)

md_loader = DirectoryLoader(
    KNOWLEDGE_BASE,
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

# pdf_loader = DirectoryLoader(
#     KNOWLEDGE_BASE,
#     glob="**/*.pdf",
#     loader_cls=PyPDFLoader
# )

documents = md_loader.load() #+ pdf_loader.load()

print("Documents loaded:", len(documents))

## Quick Check the documents

In [ ]:
print(documents[1])

## Chunk Documents

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Chunks created:", len(chunks))

## Create Huggingface embeddings
Using 384-dimension embeddings.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## Create A Vector Database

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_PATH
)

vectorstore.persist()

print("Vector database created")

## Create A Retrieval

In [ ]:
RETRIEVAL_K =10
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

## Testing Retrieval
Testing retrieval directly

In [ ]:
docs = retriever.invoke("What courses does Boldlinks offer?")

for d in docs:
    print(d.page_content[:300])
    print("-----")


## Create OpenAI LLM

In [ ]:
MODEL = "gpt-4o-mini"

llm = ChatOpenAI(
    model=MODEL,
    temperature=0
)

## Fetch Context
Fetch data from knowledge_base

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions about Raheem Yaqub and his Boldlinks Technology Solution company using the provided context..

Context:
{context}

Question:
{question}

Answer using only the context. If the answer is not in the context say you don't know.
""")

## Do Query Re-writing to improve RAG's response
This function will re write query to easily match the context better

In [ ]:
def rewrite_query(question):

    rewrite_prompt = f"""
Rewrite the following question so it is better for searching a knowledge base.

Question: {question}
"""

    response = llm.invoke(rewrite_prompt)

    return response.content

## RAG Answer Function
This answers question based on context

In [ ]:
def fetch_context(question):

    better_query = rewrite_query(question)

    docs = retriever.invoke(better_query)

    return docs

def answer_question(question):
    context = fetch_context(question)

    messages = prompt.format_messages(
        context=context,
        question=question
    )

    response = llm.invoke(messages)

    return response.content

## Test RAG
Run a simple test

In [ ]:
answer_question("What degrees does Yaqub have?")

## Another Test

In [ ]:
answer_question("What courses does Boldlinks offer?")

## Gradio Chat UI

In [ ]:
import gradio as gr

def chat(message, history):

    answer = answer_question(message)

    return answer


demo = gr.ChatInterface(
    fn=chat,
    title="Yaqub Private Assistant",
    description="Ask questions about Raheem Yaqub and his Boldlinks Technology Solution company"
)

demo.launch()